In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Load cleaned data
df = pd.read_csv('../data/online_retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)

engine = create_engine('mysql+pymysql://root:Ni20@10#2005@localhost:3306/ecommerce_analytics')

print("Connected:", engine)

Connected: Engine(mysql+pymysql://root:***@10#2005@localhost:3306/ecommerce_analytics)


In [9]:
import os
os.environ["MYSQL_PASSWORD"] = input("Enter MySQL password: ")

In [11]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=os.environ.get("MYSQL_PASSWORD"),
    host="localhost",
    port=3306,
    database="ecommerce_analytics"
)
engine = create_engine(url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


In [4]:
# 1. Load unique customers
customers = df[['CustomerID', 'Country']].drop_duplicates(subset='CustomerID')
customers.to_sql('customers', engine, if_exists='append', index=False)
print(f"Loaded {len(customers)} customers")

Loaded 4338 customers


In [5]:
# 2. Load unique products
products = df[['StockCode', 'Description']].drop_duplicates(subset='StockCode')
products = products.dropna(subset=['Description'])
products.to_sql('products', engine, if_exists='append', index=False)
print(f"Loaded {len(products)} products")

Loaded 3665 products


In [6]:
# 3. Load unique invoices
invoices = df[['InvoiceNo', 'CustomerID', 'InvoiceDate']].drop_duplicates(subset='InvoiceNo')
invoices.to_sql('invoices', engine, if_exists='append', index=False)
print(f"Loaded {len(invoices)} invoices")

Loaded 18532 invoices


In [7]:
# 4. Load invoice items (line-item level detail)
invoice_items = df[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'TotalPrice']]
invoice_items.to_sql('invoice_items', engine, if_exists='append', index=False)
print(f"Loaded {len(invoice_items)} invoice items")

Loaded 397884 invoice items
